In [1]:
"""
Coulomb Friction Analyzer
=========================
Models static and kinetic (Coulomb) friction for:
  1. Blocks on inclined planes (with an optional applied force along the incline)
  2. Wedge problems (raising a load / self-locking check)

Also plots friction force vs. applied load, showing the static region
(friction = applied load, up to the static limit) and the kinetic region
(friction plateaus at mu_k * N once sliding begins).

Sign convention: friction force magnitude is always reported as positive;
`state` tells you whether it's "static" (equilibrium holds) or
"kinetic" (sliding, block accelerates).

Dependencies:
    pip install numpy matplotlib
"""

from dataclasses import dataclass
from typing import Optional, Tuple, List
import numpy as np
import matplotlib.pyplot as plt


G = 9.81  # m/s^2  (set to 32.2 for ft/s^2 / imperial problems)


# --------------------------------------------------------------------------
# Inclined plane friction
# --------------------------------------------------------------------------

@dataclass
class InclinedPlane:
    weight: float          # W (force units, e.g. N or lb)
    theta_deg: float       # incline angle from horizontal
    mu_s: float            # coefficient of static friction
    mu_k: float            # coefficient of kinetic friction
    mass: Optional[float] = None   # only needed if you want acceleration (W = mass*g if None)

    def __post_init__(self):
        if self.mass is None:
            self.mass = self.weight / G

    def equilibrium(self, P: float = 0.0) -> dict:
        """
        Analyze a block on the incline with an applied force P acting
        UP the incline surface (P < 0 means it acts down-slope, adding
        to the driving force instead of opposing it).

        Returns a dict with normal force, required/available friction,
        state ("static" or "kinetic"), and acceleration if sliding.
        """
        theta = np.radians(self.theta_deg)
        N = self.weight * np.cos(theta)
        gravity_component = self.weight * np.sin(theta)   # down-slope, positive

        net_driving = gravity_component - P   # positive = tends to slide DOWN-slope
        f_max_static = self.mu_s * N
        f_kinetic = self.mu_k * N

        if abs(net_driving) <= f_max_static:
            state = "static"
            friction = abs(net_driving)      # friction exactly balances -> equilibrium
            acceleration = 0.0
        else:
            state = "kinetic"
            friction = f_kinetic
            # net unbalanced force along incline (magnitude), direction = same as net_driving
            net_force = abs(net_driving) - f_kinetic
            acceleration = net_force / self.mass

        return {
            "N": N,
            "gravity_component": gravity_component,
            "net_driving_force": net_driving,
            "f_max_static": f_max_static,
            "f_kinetic": f_kinetic,
            "friction": friction,
            "state": state,
            "acceleration": acceleration,
        }

    def is_self_locking(self) -> bool:
        """True if the block stays put under gravity alone (P=0)."""
        return self.mu_s >= np.tan(np.radians(self.theta_deg))

    def sweep_applied_force(self, P_min: float, P_max: float, n: int = 300):
        """
        Sweep the applied up-slope force P and return arrays of
        (P, friction_force, state) for plotting.
        """
        P_vals = np.linspace(P_min, P_max, n)
        friction_vals = np.zeros(n)
        states = []
        for i, P in enumerate(P_vals):
            r = self.equilibrium(P)
            friction_vals[i] = r["friction"]
            states.append(r["state"])
        return P_vals, friction_vals, states

    def plot_friction_vs_load(self, P_min: float = 0, P_max: Optional[float] = None,
                               n: int = 300, show=True, savepath=None):
        theta = np.radians(self.theta_deg)
        N = self.weight * np.cos(theta)
        if P_max is None:
            P_max = 2 * (self.mu_s * N + self.weight * np.sin(theta))

        P_vals, friction_vals, states = self.sweep_applied_force(P_min, P_max, n)
        static_mask = np.array([s == "static" for s in states])

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(P_vals[static_mask], friction_vals[static_mask],
                color="steelblue", lw=2.5, label="Static region (equilibrium)")
        ax.plot(P_vals[~static_mask], friction_vals[~static_mask],
                color="crimson", lw=2.5, label="Kinetic region (sliding)")

        f_max_s = self.mu_s * N
        f_k = self.mu_k * N
        ax.axhline(f_max_s, color="steelblue", ls="--", lw=1, alpha=0.6,
                   label=f"$\\mu_s N$ = {f_max_s:.2f}")
        ax.axhline(f_k, color="crimson", ls="--", lw=1, alpha=0.6,
                   label=f"$\\mu_k N$ = {f_k:.2f}")

        ax.set_xlabel("Applied load P (up-slope)")
        ax.set_ylabel("Friction force")
        ax.set_title(f"Friction vs. Applied Load  (\u03b8={self.theta_deg}\u00b0, "
                      f"\u03bc_s={self.mu_s}, \u03bc_k={self.mu_k})")
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

        if savepath:
            plt.savefig(savepath, dpi=150, bbox_inches="tight")
        if show:
            plt.show()
        plt.close()


# --------------------------------------------------------------------------
# Wedge friction
# --------------------------------------------------------------------------

def wedge_analysis(W: float, alpha_deg: float, mu1: float, mu2: float) -> dict:
    """
    Simple flat-faced wedge used to raise a load W, with friction at both
    contact surfaces:
      mu1 = friction coefficient at the block-wedge interface
      mu2 = friction coefficient at the wedge-ground interface
      alpha_deg = wedge angle

    Uses the standard friction-angle approach (phi = atan(mu)):
      P_push_in  = W * tan(alpha + phi1 + phi2)   force to drive the wedge in (raise load)
      Self-locking if alpha <= phi1 + phi2 (wedge will NOT slide back out on its own)
      P_pull_out = W * tan(phi1 + phi2 - alpha)   force needed to pull the wedge back out
                    (only meaningful/positive when self-locking)
    """
    phi1 = np.arctan(mu1)
    phi2 = np.arctan(mu2)
    alpha = np.radians(alpha_deg)

    P_push_in = W * np.tan(alpha + phi1 + phi2)
    self_locking = alpha <= (phi1 + phi2)

    if self_locking:
        P_pull_out = W * np.tan(phi1 + phi2 - alpha)
        note = "Self-locking: wedge stays in place under the load with no holding force needed."
    else:
        P_pull_out = -W * np.tan(alpha - (phi1 + phi2))
        note = "NOT self-locking: wedge will slide back out under the load alone (needs a holding force)."

    return {
        "phi1_deg": np.degrees(phi1),
        "phi2_deg": np.degrees(phi2),
        "P_push_in": P_push_in,
        "self_locking": self_locking,
        "P_pull_out": P_pull_out,
        "note": note,
    }


def plot_wedge_force_vs_angle(W: float, mu1: float, mu2: float,
                                alpha_max: float = 45.0, n: int = 300,
                                show=True, savepath=None):
    """
    Sweep the wedge angle and plot the force required to push the wedge in,
    shading the self-locking region (alpha <= phi1+phi2) vs. the region
    where the wedge would slide back out on its own.
    """
    phi1 = np.arctan(mu1)
    phi2 = np.arctan(mu2)
    alpha_lock_deg = np.degrees(phi1 + phi2)

    alphas = np.linspace(0.01, alpha_max, n)
    P_vals = np.array([wedge_analysis(W, a, mu1, mu2)["P_push_in"] for a in alphas])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(alphas, P_vals, color="black", lw=2, label="P required to push wedge in")

    ax.axvspan(0, min(alpha_lock_deg, alpha_max), color="seagreen", alpha=0.15,
               label=f"Self-locking region (\u03b1 \u2264 {alpha_lock_deg:.1f}\u00b0)")
    if alpha_lock_deg < alpha_max:
        ax.axvspan(alpha_lock_deg, alpha_max, color="crimson", alpha=0.1,
                   label="Not self-locking")

    ax.axvline(alpha_lock_deg, color="gray", ls="--", lw=1)
    ax.set_xlabel("Wedge angle \u03b1 (degrees)")
    ax.set_ylabel("Force P required to push wedge in")
    ax.set_title(f"Wedge Force vs. Angle  (\u03bc1={mu1}, \u03bc2={mu2}, W={W})")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    if savepath:
        plt.savefig(savepath, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    plt.close()


# --------------------------------------------------------------------------
# Demo
# --------------------------------------------------------------------------

if __name__ == "__main__":
    # --- Inclined plane example ---
    print("=== Block on Incline (gravity only, P=0) ===")
    plane = InclinedPlane(weight=200, theta_deg=20, mu_s=0.30, mu_k=0.25)
    print(f"Self-locking under gravity alone: {plane.is_self_locking()}")
    result = plane.equilibrium(P=0)
    for k, v in result.items():
        print(f"  {k}: {v}")
    print()

    print("=== Same incline, with applied up-slope force P=100 ===")
    result2 = plane.equilibrium(P=100)
    for k, v in result2.items():
        print(f"  {k}: {v}")
    print()

    plane.plot_friction_vs_load(P_min=0, P_max=150, show=False, savepath="friction_vs_load.png")
    print("Saved friction_vs_load.png")
    print()

    # --- Wedge example ---
    print("=== Wedge Analysis ===")
    wedge = wedge_analysis(W=500, alpha_deg=10, mu1=0.20, mu2=0.20)
    for k, v in wedge.items():
        print(f"  {k}: {v}")
    print()

    plot_wedge_force_vs_angle(W=500, mu1=0.20, mu2=0.20, alpha_max=45,
                               show=False, savepath="wedge_force_vs_angle.png")
    print("Saved wedge_force_vs_angle.png")

=== Block on Incline (gravity only, P=0) ===
Self-locking under gravity alone: False
  N: 187.9385241571817
  gravity_component: 68.40402866513374
  net_driving_force: 68.40402866513374
  f_max_static: 56.381557247154504
  f_kinetic: 46.98463103929542
  friction: 46.98463103929542
  state: kinetic
  acceleration: 1.0506214535473695

=== Same incline, with applied up-slope force P=100 ===
  N: 187.9385241571817
  gravity_component: 68.40402866513374
  net_driving_force: -31.595971334866263
  f_max_static: 56.381557247154504
  f_kinetic: 46.98463103929542
  friction: 31.595971334866263
  state: static
  acceleration: 0.0

Saved friction_vs_load.png

=== Wedge Analysis ===
  phi1_deg: 11.309932474020215
  phi2_deg: 11.309932474020215
  P_push_in: 320.00764981034496
  self_locking: True
  P_pull_out: 111.94527143077654
  note: Self-locking: wedge stays in place under the load with no holding force needed.

Saved wedge_force_vs_angle.png
